# Практикум 4. Генеративный ИИ в отчётности и управлении рисками (Google Colab)
**Принцип:** LLM готовит и структурирует, код считает и проверяет, человек утверждает. Поэтому в блокноте всё проверяемое работает **без внешних моделей**, а место LLM занимает функция-заглушка, которую вы замените локальной или разрешённой моделью.
**После блокнота вы умеете:** (1) классифицировать тексты риск-событий и решать, что отдавать человеку; (2) обернуть LLM-классификатор строгой схемой ответа и валидатором; (3) находить связанные риски и строить карту реестра; (4) автоматически проверять трассировку утверждений черновика и знать, что автоматика не ловит; (5) планировать выборку ручной проверки (интервал Уилсона) и считать согласие разметчиков (κ Коэна); (6) считать стресс-сценарии методом Монте-Карло (VaR, ES, допустимый простой) со сценариями из JSON с валидацией.
**Данные:** синтетические тексты риск-событий, реестр рисков, источники и утверждения черновика, модель потерь платёжной инфраструктуры. **Артефакты:** `p4_metrics.json`, `claims_check.csv`, `scenarios.json`, рисунки `fig_*.png`. Запуск: Среда выполнения → Выполнить всё.

In [ ]:
import numpy as np, pandas as pd, json, re
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, cohen_kappa_score
from sklearn.metrics.pairwise import cosine_similarity
SEED = 42; rng = np.random.default_rng(SEED); pd.options.display.float_format = '{:,.3f}'.format
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False,'axes.titlesize':11})
TEAL, AMBER, RED, NAVY, GREY, BLUE = '#0E7C86', '#F2A541', '#D1495B', '#0B2545', '#9AA5B1', '#5B8DB8'

## 1. Тексты риск-событий: классификация
Восемь категорий, 1600 описаний: 25% «смешанных» (фразы двух категорий), 5% шума в метках. Сплит по времени: первые 1200 — обучение, последние 400 — тест.

In [ ]:
CATS = ['Информационная безопасность','Сбой ИТ-системы','Ошибка персонала','Внешнее мошенничество','Комплаенс','Сбой поставщика','Ошибка расчётов','Чрезвычайная ситуация']
T = {CATS[0]: ['Обнаружена фишинговая рассылка сотрудникам, {n} человек перешли по ссылке','Зафиксирована попытка несанкционированного доступа к {s} с внешнего адреса','Выявлено заражение рабочих станций вредоносным ПО в {d}','Утечка учётных данных пользователей {s} в открытых источниках'],
     CATS[1]: ['Недоступность {s} в течение {h} часов из-за отказа сервера','Сбой обновления {s} привёл к остановке обработки операций','Отказ канала связи с {d}, операции обрабатывались с задержкой','Деградация производительности {s} в пиковые часы'],
     CATS[2]: ['Сотрудник {d} ошибочно провёл платёж на сумму {a} тыс. руб.','Нарушение регламента при ручном вводе данных в {s}','Недостаточная квалификация привела к ошибкам при обработке операций','Не выполнена обязательная процедура согласования операции'],
     CATS[3]: ['Клиенты получили звонки от лиц, представившихся сотрудниками банка, ущерб {a} тыс. руб.','Выявлены поддельные платёжные поручения от имени клиента','Зафиксирована схема вывода средств через подставные счета','Обнаружено использование украденных карт в {n} операциях'],
     CATS[4]: ['Нарушен срок направления отчётности регулятору','Выявлено несоответствие процедур требованиям по противодействию легализации','Не проведена идентификация клиента при открытии счёта в {d}','Предписание регулятора по итогам проверки подразделения {d}'],
     CATS[5]: ['Подрядчик не обеспечил оказание услуг в установленный срок','Отказ вендора {s} в поддержке после обновления','Просрочка поставки оборудования для {d}','Исполнитель передал данные третьим лицам без согласия'],
     CATS[6]: ['Ошибка в алгоритме расчёта комиссий привела к некорректным начислениям','Расхождения в сверке остатков между {s} и учётной системой','Неверно применены тарифы при расчёте платежей','Дублирование начислений по {n} договорам'],
     CATS[7]: ['Срабатывание пожарной сигнализации в здании {d}, эвакуация персонала','Отключение электроснабжения офиса на {h} часов','Затопление серверного помещения {d}','Повреждение кабельной магистрали при строительных работах']}
SYS, DEP, TAIL = ['АБС','ERP','СЭД','CRM','процессинга'], ['филиала','расчётного центра','департамента','колл-центра'], ['',' по предварительной оценке',' в отчётном периоде',' по данным службы мониторинга']
def fill(t): return t.format(n=rng.integers(2,40), s=rng.choice(SYS), d=rng.choice(DEP), h=rng.integers(1,12), a=rng.integers(20,900))
rows = []
for i in range(1600):
    c = rng.choice(CATS); txt = fill(rng.choice(T[c]))+rng.choice(TAIL)
    if rng.random() < .25: c2 = rng.choice([x for x in CATS if x != c]); t2 = fill(rng.choice(T[c2])); txt = (txt+'; также '+t2.lower()) if rng.random() < .5 else (t2+'; также '+txt.lower())
    lab = c if rng.random() > .05 else rng.choice(CATS); rows.append((i, c, lab, txt))
ev = pd.DataFrame(rows, columns=['id','true','label','text']); tr, te = ev.iloc[:1200], ev.iloc[1200:]
print(ev.label.value_counts().to_dict())

In [ ]:
res = {}
for name, vec in [('слова', TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True)), ('символы 2–5', TfidfVectorizer(analyzer='char_wb', ngram_range=(2,5), min_df=2, sublinear_tf=True))]:
    Xtr = vec.fit_transform(tr.text); clf = LogisticRegression(C=10, max_iter=2000).fit(Xtr, tr.label); p = clf.predict_proba(vec.transform(te.text)); pred = clf.classes_[p.argmax(1)]
    res[name] = {'accuracy': accuracy_score(te.true, pred), 'macro-F1': f1_score(te.true, pred, average='macro')}
    MODS = globals().setdefault('MODS', {}); MODS[name] = (vec, clf, p, pred)
best_name = max(res, key=lambda k: res[k]['macro-F1']); best_vec, best_clf, P_te, pred_te = MODS[best_name]
print(pd.DataFrame(res).T, '| лучшая:', best_name)
f1c = pd.Series(f1_score(te.true, pred_te, average=None, labels=CATS), CATS).round(3); print(f1c.to_dict())
conf = pd.Series(P_te.max(1)); COV = []
for th in [0, .3, .4, .5, .6, .7]:
    m = (conf >= th).values; COV.append((th, m.mean(), accuracy_score(te.true[m], pred_te[m])))
print(pd.DataFrame(COV, columns=['порог','доля авто','точность авто']))

## 2. LLM-классификатор в «строгой обёртке» (имитатор)
`fake_llm` — **имитатор**: 5% битых JSON, 4% выдуманных категорий, 12% ошибок выбора. Он нужен, чтобы отладить контур; реальные цифры вы получите, подставив свою модель. Контур: строгая схема → валидатор → повтор → отказ (на ручной разбор).

In [ ]:
VALID = set(CATS)
def fake_llm(text, true, r):
    u = r.random()
    if u < .05: return '{"category": "'+true+'", "confidence": 0.9'                          # битый JSON
    if u < .09: return json.dumps({'category': 'Кибератака', 'confidence': .8}, ensure_ascii=False)   # выдуманная категория
    cat = true if r.random() > .12 else r.choice(CATS); return json.dumps({'category': cat, 'confidence': round(float(r.uniform(.5,.99)),2)}, ensure_ascii=False)
def parse(s):
    try: o = json.loads(s)
    except Exception: return None
    return o if set(o) == {'category','confidence'} and o['category'] in VALID and 0 <= o['confidence'] <= 1 else None
def ask(text, true, r, retries=2, votes=1):
    out = []
    for _ in range(votes):
        for _ in range(retries+1):
            o = parse(fake_llm(text, true, r))
            if o: out.append(o['category']); break
    if not out: return None
    c = pd.Series(out).mode()[0]; return c if out.count(c) > votes//2 else None
r = np.random.default_rng(3); subset = te.iloc[:300]
G = {}
for lab, kw in [('без валидатора', None), ('валидатор + 2 повтора', dict(retries=2, votes=1)), ('валидатор + 3 голоса', dict(retries=2, votes=3))]:
    r = np.random.default_rng(5)
    if kw is None: pr = [(lambda s: (json.loads(s)['category'] if s.strip().endswith('}') else None))(fake_llm(t, y, r)) for t, y in zip(subset.text, subset.true)]
    else: pr = [ask(t, y, r, **kw) for t, y in zip(subset.text, subset.true)]
    ok = np.array([p_ is not None and p_ in VALID for p_ in pr]); acc_ok = np.mean([p_ == y for p_, y, o in zip(pr, subset.true, ok) if o])
    G[lab] = {'принято': ok.mean(), 'выдуманных категорий': float(np.mean([p_ is not None and p_ not in VALID for p_ in pr])), 'точность принятых': acc_ok, 'на ручной разбор': 1-ok.mean()}
print(pd.DataFrame(G).T.to_string())

## 3. Связанные риски и карта реестра
Реестр из 60 рисков: описание, категория, вероятность и ущерб (1–5). Связь — косинусное сходство TF-IDF выше порога.

In [ ]:
reg = []
for i in range(60):
    c = rng.choice(CATS); d = fill(rng.choice(T[c])).replace('Обнаружена','Возможна').replace('Выявлено','Возможно').replace('Зафиксирована','Возможна'); reg.append((i, c, d, int(np.clip(rng.normal(3,1),1,5)), int(np.clip(rng.normal(3,1.1),1,5))))
RG = pd.DataFrame(reg, columns=['rid','cat','text','lik','imp']); Sim = cosine_similarity(TfidfVectorizer(analyzer='char_wb', ngram_range=(2,5), sublinear_tf=True).fit_transform(RG.text)); np.fill_diagonal(Sim, 0)
thr = np.quantile(Sim[np.triu_indices(60,1)], .96); edges = [(i,j,Sim[i,j]) for i in range(60) for j in range(i+1,60) if Sim[i,j] >= thr]
cross = [e for e in edges if RG.cat[e[0]] != RG.cat[e[1]]]; print('порог', round(thr,3), '| связей', len(edges), '| между категориями', len(cross)); print(RG.assign(score=RG.lik*RG.imp).sort_values('score', ascending=False).head(3)[['rid','cat','lik','imp']])

## 4. Трассировка утверждений черновика отчёта
У каждого утверждения должна быть ссылка `[S№]` на источник; число и направление изменения должны совпадать. Автоматика ловит ссылки, числа, направление и причинные слова; **охват** («по подразделению» → «по банку») она не ловит — это работа выборки и человека.

In [ ]:
SRC = {f'S{i}': dict(metric=m, value=round(float(v),1), unit=u, trend=tn, scope=sc, cause=None) for i,(m,v,u,tn,sc) in enumerate(
    [(rng.choice(['Число инцидентов ИБ','Время простоя АБС','Доля ошибок в расчётах','Число жалоб клиентов','Потери от мошенничества','Число нарушений регламента']), rng.uniform(1,90), rng.choice(['шт.','ч.','%','млн руб.']), rng.choice(['вырос','снизился']), rng.choice(['по подразделению А','по филиалу Б','по расчётному центру'])) for _ in range(40)], start=1)}
def make_claim(sid, kind):
    s = SRC[sid]; v = s['value']; tn = s['trend']; sc = s['scope']; cite = f'[{sid}]'; extra = ''
    if kind == 'число': v = round(v*rng.choice([.7,1.4,1.8]),1)
    if kind == 'направление': tn = 'снизился' if tn == 'вырос' else 'вырос'
    if kind == 'охват': sc = 'по банку в целом'
    if kind == 'причина': extra = ', вследствие ошибок персонала'
    if kind == 'ссылка': cite = f'[S{rng.integers(60,99)}]'
    if kind == 'без ссылки': cite = ''
    return f"{s['metric']} {sc} {tn} до {v} {s['unit']}{extra} {cite}".strip()
KINDS = ['верно']*120 + [k for k in ['число','направление','охват','причина','ссылка','без ссылки'] for _ in range(13)]
cl = pd.DataFrame({'kind': KINDS, 'sid': rng.choice(list(SRC), len(KINDS))}); cl['text'] = [make_claim(s, k) for s, k in zip(cl.sid, cl.kind)]
def check(text):
    flags = []; m = re.search(r'\[(S\d+)\]', text)
    if not m: return ['без ссылки']
    if m.group(1) not in SRC: return ['ссылка']
    s = SRC[m.group(1)]; nums = [float(x) for x in re.findall(r'до ([\d.]+)', text)]
    if nums and abs(nums[0]-s['value']) > 0.05: flags.append('число')
    if s['trend'] not in text: flags.append('направление')
    if re.search(r'вследствие|из-за|в результате', text) and not s['cause']: flags.append('причина')
    return flags
cl['flg'] = cl.text.map(check); cl['flagged'] = cl.flg.map(bool)
REC = {k: float(cl[cl.kind == k].flagged.mean()) for k in ['число','направление','причина','ссылка','без ссылки','охват']}; FPR = float(cl[cl.kind == 'верно'].flagged.mean())
print('полнота по типам ошибок:', {k: round(v,2) for k,v in REC.items()}, '| ложные срабатывания на верных:', round(FPR,3)); print(cl[cl.kind=='охват'].text.iloc[0]); cl.to_csv('claims_check.csv', index=False)

## 5. Сколько проверять вручную и согласие разметчиков
Интервал Уилсона для доли ошибок: `(p + z²/2n ± z·√(p(1−p)/n + z²/4n²)) / (1 + z²/n)`. κ Коэна — согласие двух проверяющих сверх случайного.

In [ ]:
def wilson(k, n, z=1.96):
    p = k/n; d = 1+z*z/n; c = (p+z*z/(2*n))/d; h = z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/d; return c-h, c+h
print('28 из 30 подтверждено:', [round(x,3) for x in wilson(28,30)], '| 4 ошибки из 50:', [round(x,3) for x in wilson(4,50)])
W = {n: wilson(round(.1*n), n) for n in [20,30,50,100,200,400]}; print({n: round(h-l,3) for n,(l,h) in W.items()})
truth = (cl.kind != 'верно').values.astype(int); fl = lambda err: np.where(rng.random(len(truth)) < err, 1-truth, truth)
A, B = fl(.08), fl(.10); kappa = cohen_kappa_score(A, B); tab = pd.crosstab(A, B); print('κ =', round(kappa,3)); print(tab)
BAD = {'всегда': 'часто', 'гарантированно': 'с высокой вероятностью', 'полностью исключено': 'маловероятно', 'абсолютно': '', 'очевидно': ''}
dr = ['Риск мошенничества полностью исключено благодаря новым контролям.', 'Потери всегда оставались в пределах лимита.', 'Контроль гарантированно предотвращает сбои.', 'Время простоя составило 3,2 ч. [S4].', 'Число инцидентов снизилось до 14 шт. [S2].']*20
lint = lambda t: [w for w in BAD if w in t]; fl_ = [lint(t) for t in dr]; print('линтер: помечено', sum(map(bool, fl_)), 'из', len(dr), '| пример:', dr[0], '→', fl_[0])

## 6. Монте-Карло: потери платёжной инфраструктуры
Годовые потери = сумма по отказам компонентов: число отказов ~ Пуассон(λ·m), общий фактор `m = exp(ρZ − ρ²/2)`; длительность простоя — логнормальная; отказ failover утраивает простой ЦОД. Сценарии — параметры из JSON после валидации.

In [ ]:
COMP = ['ЦОД-1','ЦОД-2','Канал связи','Процессинг (вендор)','Платёжный шлюз']
BASE = dict(lam=np.array([.8,.8,2.5,1.2,1.0]), mu=np.log([2.5,2.5,1.2,3.0,1.5]), sig=np.array([.8,.8,.7,.9,.8]), rho=.4, L=4.0, fixed=.3, q=.1)
def simulate(par, N, seed):
    r = np.random.default_rng(seed); m = np.exp(par['rho']*r.standard_normal(N)-par['rho']**2/2); loss = np.zeros(N); worst = np.zeros(N)
    for i in range(len(par['lam'])):
        k = r.poisson(par['lam'][i]*m); tot = k.sum(); idx = np.repeat(np.arange(N), k)
        h = np.exp(par['mu'][i]+par['sig'][i]*r.standard_normal(tot))
        if i < 2: h = np.where(r.random(tot) < par['q'], h*3, h)
        loss += np.bincount(idx, weights=h*par['L']+par['fixed'], minlength=N); np.maximum.at(worst, idx, h)
    return loss, worst
def metrics(loss, worst, tol=4.0):
    v99 = np.quantile(loss,.99); return {'mean': loss.mean(), 'VaR95': np.quantile(loss,.95), 'VaR99': v99, 'VaR99.9': np.quantile(loss,.999), 'ES99': loss[loss >= v99].mean(), 'P(простой > %.0f ч)' % tol: float((worst > tol).mean())}
base_loss, base_worst = simulate(BASE, 300_000, 7); M0 = metrics(base_loss, base_worst); print({k: round(v,2) for k,v in M0.items()})

In [ ]:
LIM = {'lam_mult': (.5, 5), 'mu_shift': (-.5, 1.0), 'rho': (0, 1.2), 'q': (0, .8), 'L_mult': (.5, 3)}
def validate(sc):
    ok = set(sc) <= set(LIM)|{'name','target'} and sc.get('target','all') in COMP+['all'] and all(LIM[k][0] <= v <= LIM[k][1] for k, v in sc.items() if k in LIM); return ok
def apply(par, sc):
    p = {k: (v.copy() if hasattr(v,'copy') else v) for k, v in par.items()}; ix = list(range(len(COMP))) if sc.get('target','all') == 'all' else [COMP.index(sc['target'])]
    for i in ix: p['lam'][i] *= sc.get('lam_mult',1); p['mu'][i] += sc.get('mu_shift',0)
    for k in ['rho','q']: p[k] = sc.get(k, p[k])
    p['L'] *= sc.get('L_mult',1); return p
SC = [{'name':'Отказ вендора процессинга','target':'Процессинг (вендор)','lam_mult':3,'mu_shift':.4}, {'name':'Каскад ЦОД','target':'all','q':.5,'lam_mult':1.5}, {'name':'Кибератака (общий фактор)','target':'all','rho':.9,'lam_mult':1.5}, {'name':'Выдуманный сценарий LLM','target':'all','lam_mult':12}]
print([(s['name'], validate(s)) for s in SC]); SC = [s for s in SC if validate(s)]
S = {'база': M0}; LS = {'база': base_loss}
for s in SC: l, w = simulate(apply(BASE, s), 300_000, 7); S[s['name']] = metrics(l, w); LS[s['name']] = l
print(pd.DataFrame(S).T.round(1)); json.dump(SC, open('scenarios.json','w'), ensure_ascii=False, indent=1)

In [ ]:
CONV = {}
for N in [1000, 3000, 10_000, 30_000, 100_000]:
    v = [np.quantile(simulate(BASE, N, 100+s)[0], .99) for s in range(20)]; CONV[N] = (float(np.median(v)), float(np.min(v)), float(np.max(v)))
print({N: round((hi-lo)/md_,2) for N, (md_, lo, hi) in CONV.items()})
TOR = {}
for name, f in [('λ ЦОД', lambda p,s: p['lam'].__setitem__(slice(0,2), p['lam'][:2]*s)), ('λ канал', lambda p,s: p['lam'].__setitem__(2, p['lam'][2]*s)), ('λ вендор', lambda p,s: p['lam'].__setitem__(3, p['lam'][3]*s)), ('λ шлюз', lambda p,s: p['lam'].__setitem__(4, p['lam'][4]*s)),
                ('ρ общий фактор', lambda p,s: p.__setitem__('rho', p['rho']*s)), ('σ простоя', lambda p,s: p.__setitem__('sig', p['sig']*s)), ('стоимость часа L', lambda p,s: p.__setitem__('L', p['L']*s)), ('q отказа failover', lambda p,s: p.__setitem__('q', p['q']*s))]:
    out = []
    for s in (.8, 1.2): p = {k: (v.copy() if hasattr(v,'copy') else v) for k, v in BASE.items()}; f(p, s); out.append(np.quantile(simulate(p, 100_000, 1)[0], .99))
    TOR[name] = out
b99 = np.quantile(simulate(BASE, 100_000, 1)[0], .99); print({k: [round(x-b99,1) for x in v] for k, v in TOR.items()})

## 7. Рисунки для презентации

In [ ]:
def style(ax): ax.grid(alpha=.25, lw=.5); ax.set_axisbelow(True)
# 1. Матрица ошибок
cm = confusion_matrix(te.true, pred_te, labels=CATS); fig, ax = plt.subplots(figsize=(5.4,3.9)); ax.imshow(cm, cmap='Blues'); short = ['ИБ','ИТ','Персонал','Мошенн.','Комплаенс','Поставщик','Расчёты','ЧС']
ax.set_xticks(range(8)); ax.set_xticklabels(short, rotation=45, ha='right'); ax.set_yticks(range(8)); ax.set_yticklabels(short)
for i in range(8):
    for j in range(8): ax.text(j, i, cm[i,j], ha='center', va='center', color='white' if cm[i,j] > cm.max()/2 else NAVY, fontsize=8)
ax.set_xlabel('предсказано'); ax.set_ylabel('истина'); ax.set_title('Матрица ошибок, тест (400 описаний)'); fig.tight_layout(); fig.savefig('fig_confusion.png', dpi=200)
# 2. Сеть связанных рисков
fig, ax = plt.subplots(figsize=(4.4,3.5)); ang = np.linspace(0, 2*np.pi, 9)[:-1]; cen = {c: (np.cos(a), np.sin(a)) for c, a in zip(CATS, ang)}
pos = np.array([[cen[c][0]+rng.normal(0,.2), cen[c][1]+rng.normal(0,.2)] for c in RG.cat]); cmap = plt.cm.tab10(np.arange(8))
for i, j, s_ in edges: ax.plot(pos[[i,j],0], pos[[i,j],1], c=AMBER if RG.cat[i] != RG.cat[j] else GREY, lw=1.6 if RG.cat[i] != RG.cat[j] else .8, alpha=.8, zorder=1)
ax.scatter(pos[:,0], pos[:,1], s=60, c=[cmap[CATS.index(c)] for c in RG.cat], zorder=2, edgecolor='white'); ax.set_axis_off(); ax.set_title('Связанные риски: янтарные связи между категориями'); fig.tight_layout(); fig.savefig('fig_network.png', dpi=200)
# 3. Тепловая карта реестра
fig, ax = plt.subplots(figsize=(4.4,3.5)); g = RG.groupby(['imp','lik']).size().unstack(fill_value=0).reindex(index=range(1,6), columns=range(1,6), fill_value=0); zone = np.add.outer(np.arange(1,6), np.arange(1,6))
ax.imshow(zone, cmap='RdYlGn_r', origin='lower', alpha=.55, extent=(.5,5.5,.5,5.5))
for i in range(1,6):
    for j in range(1,6):
        if g.loc[i,j]: ax.scatter(j, i, s=g.loc[i,j]*90, c=NAVY, alpha=.85); ax.text(j, i, g.loc[i,j], color='white', ha='center', va='center', fontsize=8)
ax.set_xlabel('вероятность'); ax.set_ylabel('ущерб'); ax.set_title('Карта реестра: 60 рисков'); fig.tight_layout(); fig.savefig('fig_heatmap.png', dpi=200)
# 4. Ширина интервала Уилсона
fig, ax = plt.subplots(figsize=(5.4,3.2)); ns = np.arange(10, 401, 5)
for p_, c in [(.05, TEAL), (.1, AMBER), (.2, RED)]: ax.plot(ns, [wilson(round(p_*n), n)[1]-wilson(round(p_*n), n)[0] for n in ns], c=c, lw=2, label=f'доля ошибок {p_:.0%}')
ax.set_xlabel('размер выборки для ручной проверки'); ax.set_ylabel('ширина 95% интервала'); ax.set_title('Сколько утверждений проверять'); ax.legend(frameon=False); style(ax); fig.tight_layout(); fig.savefig('fig_wilson.png', dpi=200)
# 5. Распределение потерь
fig, ax = plt.subplots(figsize=(5.8,3.9)); ax.hist(np.clip(base_loss, 0, np.quantile(base_loss,.9995)), bins=80, color=TEAL, alpha=.85)
for q_, lab, c in [(M0['VaR95'],'VaR95',AMBER), (M0['VaR99'],'VaR99',RED), (M0['ES99'],'ES99',NAVY)]: ax.axvline(q_, c=c, ls='--', lw=1.6); ax.text(q_, ax.get_ylim()[1]*.92, f'{lab}\n{q_:.0f}', color=c, ha='left', fontsize=9)
ax.set_xlabel('годовые потери, млн'); ax.set_ylabel('число симуляций'); ax.set_title('Годовые потери, 300 тыс. симуляций'); style(ax); fig.tight_layout(); fig.savefig('fig_losses.png', dpi=200)
# 6. Сходимость
fig, ax = plt.subplots(figsize=(5.8,3.9)); xs = list(CONV); md_ = [CONV[n][0] for n in xs]; ax.fill_between(xs, [CONV[n][1] for n in xs], [CONV[n][2] for n in xs], color=TEAL, alpha=.25, label='разброс 20 запусков'); ax.plot(xs, md_, 'o-', c=TEAL, label='медиана VaR99')
ax.axhline(M0['VaR99'], c=RED, ls='--', label='оценка на 300 тыс.'); ax.set_xscale('log'); ax.set_xlabel('число симуляций'); ax.set_ylabel('VaR99, млн'); ax.set_title('Сходимость VaR99'); ax.legend(frameon=False); style(ax); fig.tight_layout(); fig.savefig('fig_convergence.png', dpi=200)
# 7. Сценарии
fig, ax = plt.subplots(figsize=(5.8,3.9)); names = list(S); x = np.arange(len(names)); ax.bar(x-.2, [S[n]['VaR99'] for n in names], .4, color=TEAL, label='VaR99'); ax.bar(x+.2, [S[n]['ES99'] for n in names], .4, color=AMBER, label='ES99')
ax.set_xticks(x); ax.set_xticklabels([n.replace(' (','\n(').replace('Отказ вендора процессинга','Отказ вендора') for n in names], fontsize=8); ax.set_ylabel('млн'); ax.set_title('Базовый и стрессовые сценарии'); ax.legend(frameon=False); style(ax); fig.tight_layout(); fig.savefig('fig_scenarios.png', dpi=200)
# 8. Торнадо
fig, ax = plt.subplots(figsize=(5.8,3.9)); order = sorted(TOR, key=lambda k: abs(TOR[k][1]-TOR[k][0])); 
for i, k in enumerate(order): ax.barh(i, TOR[k][0]-b99, color=BLUE, height=.6); ax.barh(i, TOR[k][1]-b99, color=AMBER, height=.6)
ax.set_yticks(range(len(order))); ax.set_yticklabels(order); ax.axvline(0, c=NAVY, lw=1); ax.set_xlabel('изменение VaR99, млн (синий −20%, янтарный +20%)'); ax.set_title('Чувствительность VaR99'); style(ax); fig.tight_layout(); fig.savefig('fig_tornado.png', dpi=200)

In [ ]:
json.dump({'classifier': res, 'f1_by_category': f1c.to_dict(), 'coverage': COV, 'llm_wrapper_simulated': G, 'register_edges': {'total': len(edges), 'cross_category': len(cross)}, 'claims_recall': REC, 'claims_fpr': FPR, 'kappa': float(kappa),
           'mc_base': {k: float(v) for k, v in M0.items()}, 'mc_scenarios': {k: {a: float(b) for a, b in v.items()} for k, v in S.items()}, 'mc_convergence': {str(k): v for k, v in CONV.items()}, 'tornado': TOR}, open('p4_metrics.json','w'), indent=1, ensure_ascii=False, default=float)

## Задания
1. Замените `fake_llm` на вызов локальной модели и повторите блок 2: сколько ответов отклоняет валидатор?
2. Добавьте в `check` проверку охвата (слова «по подразделению», «по банку») и измерьте полноту по этому типу.
3. Выберите размер выборки ручной проверки так, чтобы интервал для доли ошибок 10% был уже 0,10.
4. Добавьте свой сценарий в JSON, пропустите через `validate` и посчитайте VaR99 и ES99.
5. Увеличьте допустимый простой с 4 до 6 часов: как меняется P(простой > порога) по сценариям?